# SVM kernels, projections, and valid model selection

Use the local Sonar data: 208 observations, 60 features, rocks (R) and mines (M).
Learning outcomes: explain kernel boundaries, distinguish a 2D teaching projection
from the full-feature prediction problem, and fit preprocessing within CV folds.
Row holdouts are a classroom fallback; acquisition grouping is not supplied.
Source: https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks

Solution edition. Reference code is provided; complete the prediction, verification, controlled experiment, and interpretation tasks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "curriculum" / "build.py").exists())
SEED = 42

In [ ]:
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
folder = ROOT / "machine_learning_course/Module_1_Supervised_Learning_1/exercises/03_svm"
df = pd.read_csv(folder / "SVM_dataset.csv")
X = df.iloc[:, :-1].to_numpy(dtype=float)
# Explicit mapping avoids silently reversing the physical label meanings.
y = df.iloc[:, -1].map({"R": 0, "M": 1}).to_numpy()
assert X.shape == (208, 60) and np.isfinite(y).all()
print(df.iloc[:, -1].value_counts())
print("Label encoding: R (rock) = 0, M (mine) = 1")

## 1. Predict before running
Why would fitting PCA on all observations before cross-validation leak information, even though PCA does not use labels?

INSTRUCTOR ANSWER: Its mean and directions would depend on the evaluation inputs. A predictive CV fold must fit the scaler and PCA using its own training rows. A labelled exploratory display of one fixed dataset answers a different question.

In [ ]:
# Three partitions before fitting any scaler or PCA.
dev_idx, final_idx = train_test_split(np.arange(len(y)), test_size=0.2,
                                     stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.25,
                                    stratify=y[dev_idx], random_state=SEED)
assert not set(train_idx) & set(val_idx)
assert not set(dev_idx) & set(final_idx)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print("Train / validation / reserved final:", len(train_idx), len(val_idx), len(final_idx))

## 2. A two-dimensional teaching model
Fit the projection using training rows only. Inspect the retained variance. Do not treat two principal components as an information-preserving representation of all 60 features.

In [ ]:
projection = Pipeline([("scale", StandardScaler()), ("pca", PCA(n_components=2))])
Z_train = projection.fit_transform(X_train)
Z_val = projection.transform(X_val)
print("Variance retained:", projection.named_steps["pca"].explained_variance_ratio_.sum())
np.testing.assert_allclose(projection.named_steps["scale"].mean_, X_train.mean(axis=0))
candidates = {name: SVC(kernel=name, C=1, gamma="scale", degree=3)
              for name in ["linear", "poly", "rbf"]}
for model in candidates.values():
    model.fit(Z_train, y_train)

## 3. Compare boundaries on the same coordinates
Before plotting, predict which model can form curved boundaries. Then explain one difference using the model mechanism, not just the score.

In [ ]:
all_z = np.vstack([Z_train, Z_val])
xx, yy = np.meshgrid(np.linspace(all_z[:, 0].min()-1, all_z[:, 0].max()+1, 220),
                     np.linspace(all_z[:, 1].min()-1, all_z[:, 1].max()+1, 180))
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, candidates.items()):
    pred = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, pred, levels=[-.5, .5, 1.5], colors=["#cee2f2", "#f4dbc9"])
    for value, label, color in [(0, "Rock", "#0964a8"), (1, "Mine", "#aa4c13")]:
        rows = y_val == value
        ax.scatter(Z_val[rows, 0], Z_val[rows, 1], c=color, edgecolors="white", label=label)
    score = accuracy_score(y_val, model.predict(Z_val))
    ax.set(title=f"{name}: validation accuracy {score:.2f}", xlabel="PC 1")
axes[0].set_ylabel("PC 2")
axes[-1].legend()
fig.suptitle("Train-fitted 2D boundaries; markers are validation observations")
fig.tight_layout()
plt.show()

## 4. Tune a full-feature predictive pipeline
Use the same raw training rows. Scaling now refits inside every CV fold. The full-feature model is a different representation from the 2D demonstration. Validation comparison does not open the reserved final holdout.

In [ ]:
pipeline = Pipeline([("scale", StandardScaler()), ("svc", SVC())])
grid = [{"svc__kernel": ["linear"], "svc__C": [0.1, 1, 10]},
        {"svc__kernel": ["rbf"], "svc__C": [0.1, 1, 10],
         "svc__gamma": ["scale", 0.01, 0.1]}]
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
search = GridSearchCV(pipeline, grid, cv=cv, scoring="balanced_accuracy", n_jobs=1)
search.fit(X_train, y_train)
print("Selected parameters:", search.best_params_)
print("Selected CV balanced accuracy:", search.best_score_)
print(classification_report(y_val, search.predict(X_val),
                            target_names=["Rock", "Mine"], zero_division=0))

## 5. Controlled experiment
Add PCA to the CV pipeline and compare 2, 10, and 30 components under the same folds. Write a hypothesis before running. Report validation evidence, explain the variance/prediction distinction, and select a procedure before final assessment.

INSTRUCTOR ANSWER: Add ("pca", PCA()) between scale and svc, and pca__n_components to the search grid. Fit on X_train. Every fold then fits its own scale and PCA. High retained input variance does not guarantee preservation of the classification signal.

## 6. Locked final assessment
Set chosen to the selected fitted pipeline. After this report, do not tune further using these labels.

In [ ]:
chosen = search.best_estimator_
RUN_FINAL = False
if RUN_FINAL:
    print(classification_report(y[final_idx], chosen.predict(X[final_idx]),
                                target_names=["Rock", "Mine"], zero_division=0))

## Submission
Submit the projection figure, a description of what each split was used for, a matched model comparison, and one failure interpretation. A small row holdout cannot establish performance on a new acquisition campaign.